In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.preprocessing import LabelEncoder


In [2]:
df = pd.read_csv('surat_uncleaned.csv')  


print(df.head())
print(df.info())
null_counts = df.isnull().sum()
print(null_counts)

                                       property_name areaWithType square_feet  \
0          2 BHK Apartment for Sale in Dindoli Surat  Carpet Area    644 sqft   
1           2 BHK Apartment for Sale in Althan Surat   Super Area   1278 sqft   
2          2 BHK Apartment for Sale in Pal Gam Surat   Super Area   1173 sqft   
3     2 BHK Apartment for Sale in Jahangirabad Surat  Carpet Area    700 sqft   
4  2 BHK Apartment for Sale in Orchid Fantasia, P...   Super Area   1250 sqft   

       transaction            status        floor      furnishing  \
0     New Property  Poss. by Oct '24  5 out of 10     Unfurnished   
1     New Property  Poss. by Jan '26  6 out of 14     Unfurnished   
2           Resale     Ready to Move  5 out of 13  Semi-Furnished   
3     New Property     Ready to Move  6 out of 14     Unfurnished   
4  Orchid Fantasia      New Property  Unfurnished               2   

        facing                                        description  \
0         West  Luxury projec

In [3]:
df.replace(['-', 'NA', 'N/A', '?', 'xx', '--', 'nan'], np.nan, inplace=True)


def clean_number(col):
    return (
        df[col]
        .astype(str)
        .str.replace('[^0-9.]', '', regex=True) 
        .str.strip()
    )

num_cols = ['square_feet', 'price_per_sqft', 'price']

for col in num_cols:
    df[col] = clean_number(col)
    df[col] = pd.to_numeric(df[col], errors='coerce')
    

In [4]:
df['floor_cleaned'] = (
    df['floor']
    .astype(str)
    .str.extract('(\d+)') 
    .astype(float)
)
df['floor_cleaned'] = df['floor_cleaned'].replace({
    'Basement': 0,
    'basement': 0,
    'Basement Floor': 0,
    'Ground': 1,
    'Ground Floor': 1,
    'ground': 1,
})



In [5]:
valid_furnishing = ['Unfurnished', 'Semi-Furnished', 'Furnished']
valid_facings = [
    'East', 'West', 'North', 'South',
    'North-East', 'South-East', 'North-West', 'South-West',
    'Main Road', 'Garden Facing'
]
valid_transactions = ['New Property', 'Resale']
valid_status = ['Ready to Move', 'Under Construction']

df['transaction'] = df['transaction'].where(df['transaction'].isin(valid_transactions), np.nan)

def clean_row(row):
    f = str(row['furnishing']).strip()
    fa = str(row['facing']).strip()

    if f not in valid_furnishing:
        if f in valid_facings:
            row['facing'] = f
            row['furnishing'] = None
        elif f.replace('.', '', 1).isdigit() or len(f) < 3:
            row['furnishing'] = None
        elif ' ' in f or (f.isalpha() and f not in valid_furnishing):
            row['furnishing'] = None

    if fa not in valid_facings:
        if fa in valid_furnishing:
            row['furnishing'] = fa
            row['facing'] = None
        elif fa.replace('.', '', 1).isdigit() or len(fa) < 3:
            row['facing'] = None
        elif ' ' in fa or (fa.isalpha() and fa not in valid_facings):
            row['facing'] = None

    return row

df = df.apply(clean_row, axis=1)

def clean_status(value):
    value = str(value).strip()
    if value in valid_status:
        return value
    elif value.startswith("Poss. by"):
        return value
    else:
        return np.nan

df['status'] = df['status'].apply(clean_status)

df.reset_index(drop=True, inplace=True)
df.loc[df['furnishing'] == 'Garden/Park', ['furnishing', 'facing']] = [np.nan, 'Garden Facing']
null_counts = df.isnull().sum()
print(null_counts)

property_name        0
areaWithType         0
square_feet          6
transaction        842
status             330
floor               45
furnishing        1255
facing            2560
description       1371
price_per_sqft     368
price              173
floor_cleaned      839
dtype: int64


In [6]:

cat_cols = ['transaction', 'status', 'furnishing', 'facing']
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

df.dropna(subset=['price'], inplace=True)
df.dropna(subset=['square_feet'], inplace=True)

df['price_per_sqft'] = df['price_per_sqft'].fillna(df['price_per_sqft'].median())
df['floor_cleaned'] = df['floor_cleaned'].fillna(df['floor_cleaned'].median())

df.drop(columns=['description'], inplace=True, errors='ignore')
df.drop(columns=['floor'], inplace=True, errors='ignore')
df['floor_cleaned'] = df['floor_cleaned'].astype(int)
print(df['floor_cleaned'].unique())


[ 5  6  7  3  1  4  2  9 12 10 13 14  8 16 11 59 15 18 20 17 19]


In [7]:
print(df['facing'].head(50))
print(df.value_counts().sum())
null_counts = df.isnull().sum()
print(null_counts)
print(df.describe())
print(df['square_feet'].unique()[:20])
df.rename(columns={
    'price': 'price (₹ Lac)',
    'price_per_sqft': 'price_per_sqft (₹)'
}, inplace=True)
print(df.info())



0            West
1            East
2            East
3            East
4            East
5            East
6            East
7            East
8            East
9            East
10           East
11           East
12           East
13           East
14           East
15           East
16           East
17           East
18           East
19           East
20           East
21           West
22           East
23           West
24           East
25           East
26           East
27           East
28           East
29           East
30           East
31           East
32           East
33           East
34           East
35           East
36           East
37    Garden/Park
38           East
39           East
40           East
41      Main Road
42           East
43           East
44           East
45           East
46           East
47           East
48           East
49           East
Name: facing, dtype: object
4346
property_name     0
areaWithType      0
square_feet       0
transac

In [8]:
le = LabelEncoder()
df['furnishing_encoded'] = le.fit_transform(df['furnishing'])

df = pd.get_dummies(df, columns=['transaction'], drop_first=True)

print(df.head())

                                       property_name areaWithType  \
0          2 BHK Apartment for Sale in Dindoli Surat  Carpet Area   
1           2 BHK Apartment for Sale in Althan Surat   Super Area   
2          2 BHK Apartment for Sale in Pal Gam Surat   Super Area   
3     2 BHK Apartment for Sale in Jahangirabad Surat  Carpet Area   
4  2 BHK Apartment for Sale in Orchid Fantasia, P...   Super Area   

   square_feet            status      furnishing facing  price_per_sqft (₹)  \
0        644.0  Poss. by Oct '24     Unfurnished   West              2891.0   
1       1278.0  Poss. by Jan '26     Unfurnished   East              3551.0   
2       1173.0     Ready to Move  Semi-Furnished   East              3800.0   
3        700.0     Ready to Move     Unfurnished   East              3966.0   
4       1250.0     Ready to Move     Unfurnished   East              3600.0   

   price (₹ Lac)  floor_cleaned  furnishing_encoded  transaction_Resale  
0           33.8              5     

In [9]:
df = df.drop_duplicates()
df.info()
print(df.head(50))



<class 'pandas.core.frame.DataFrame'>
Index: 4212 entries, 0 to 4517
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   property_name       4212 non-null   object 
 1   areaWithType        4212 non-null   object 
 2   square_feet         4212 non-null   float64
 3   status              4212 non-null   object 
 4   furnishing          4212 non-null   object 
 5   facing              4212 non-null   object 
 6   price_per_sqft (₹)  4212 non-null   float64
 7   price (₹ Lac)       4212 non-null   float64
 8   floor_cleaned       4212 non-null   int64  
 9   furnishing_encoded  4212 non-null   int64  
 10  transaction_Resale  4212 non-null   bool   
dtypes: bool(1), float64(3), int64(2), object(5)
memory usage: 366.1+ KB
                                        property_name areaWithType  \
0           2 BHK Apartment for Sale in Dindoli Surat  Carpet Area   
1            2 BHK Apartment for Sale in Althan 